`BaseModel` defines schemas and enforces data validation. 

**Note:** This automatically performs the type casting for different data types.

In [1]:
from pydantic import BaseModel 

class User(BaseModel):
    id: int
    name: str
    is_active: bool = True

u = User(id='12', name="Ada")
# if we were to pass a string in id then it does throw an eror
print(repr(u))
print(type(u.id))

User(id=12, name='Ada', is_active=True)
<class 'int'>


In [2]:
from pydantic import ValidationError

try:
    user = User(id='kl',name=None)
except ValidationError as e:
    print(e.error_count(),'problems')

    for err in e.errors():
        print(f"  field={err['loc']}  message={err['msg']!r}  type={err['type']}")


2 problems
  field=('id',)  message='Input should be a valid integer, unable to parse string as an integer'  type=int_parsing
  field=('name',)  message='Input should be a valid string'  type=string_type


For stricter validation

In [3]:
from pydantic import ValidationError

try:
    user = User.model_validate(
        {'id': '43', 'name': 'Ada'},
        strict=True
    )
except ValidationError as e:
    print(e.errors()[0]['msg'])

Input should be a valid integer


## 2 · Constraints with `Field` and `Annotated`
Declare a reusable type **once**, use it everywhere (this is how TeamDesk defines `Title`, `Password`, ...).

In [4]:
from typing import Annotated 
from pydantic import EmailStr, Field, StringConstraints
# StringConstraints is used for enforcing rules and automatic formatting on text on str fields
# EmailStr validates if input string is properly formatted emailed address  

Username = Annotated[str, StringConstraints(strip_whitespace=True,
                                            min_length=3,
                                            max_length=20,
                                            pattern=r"^[a-z0-9_]+$")]

class SignUp(BaseModel):
    username: Username
    age: int = Field(ge=13, le=120, description="Age in years")
    email: EmailStr

good = SignUp(username="ramsay23", age=32, email='ramsay@gmail.com')
print(good)

for bad in [dict(username="A!", age=5, email="nope")]:
    try: SignUp(**bad)
    except ValidationError as e:
        for err in e.errors(): print(err["loc"], "->", err["msg"])

username='ramsay23' age=32 email='ramsay@gmail.com'
('username',) -> String should have at least 3 characters
('age',) -> Input should be greater than or equal to 13
('email',) -> value is not a valid email address: An email address must have an @-sign.


## 3 · Defaults — use `default_factory` for anything that must be fresh per object

So whenever we call new instance, it loads a fresh and empty data structure of `default_factory` which can be list, dict, etc

In [5]:
from datetime import UTC, datetime 

class Note(BaseModel):
    text: str 
    tags: list[str] = Field(default_factory=list)
    created_date: datetime = Field(default_factory=lambda: datetime.now(),)


a, b = Note(text='a'), Note(text='b')
a.tags.append('x')
print(a.tags,b.tags)

print(a.created_date < b.created_date)

['x'] []
True


## 4 · Nested models and JSON
Models contain models. Errors tell you the exact *path* to the problem.

In [6]:
class Address(BaseModel):
    city: str
    country: str = "FR"

class Customer(BaseModel):
    name: str 
    addresses: list[Address]

raw_json = '''{
    "name": "Li",
    "addresses":[{"city": "Paris"}, {"city": "Tokyo", "country": "JP"}]
}'''

c = Customer.model_validate_json(raw_json) 
# when model_validate_json() we have to pass the json string 
print(c.addresses[1].country)

try:
    Customer.model_validate(
        {
            "name": "Li",
            "addresses":[{"city": "Paris"}, {"country": "JP"}]
        }
    )

except ValidationError as e:
    print(e.errors()[0]["loc"])   # ('addresses', 1, 'city')  <- second address is missing 'city'

JP
('addresses', 1, 'city')


## 5 · Validators — your own rules
* `field_validator` checks / cleans **one field**.
* `model_validator` checks **several fields together**.
* `mode="before"` runs on raw input, `mode="after"` runs on the already-typed value.

`@classmethod` is a decorator that allows to work with a class rather than the object of the class.

```
class Student:
    school = "SUTD"

    @classmethod
    def get_school(cls):
        return cls.school


print(Student.get_school())
```

- `cls` refers to `Student`
- `cls.school` accesses the class variable `school`

In [7]:
from datetime import date
from pydantic import field_validator, model_validator
from typing import Self

class Booking(BaseModel):
    start: date 
    end: date 
    guests: int 
    tags: list[str] = []

    @field_validator('tags', mode='before')
    @classmethod 
    def split_csv(cls, v):
        return [t.strip() for t in v.split(',')] if isinstance(v, str) else v

    @field_validator('guests')
    @classmethod 
    def positive(cls, v: int) -> int:
        if v < 1:
            raise ValueError('need atleast one guest')
        return v

    @model_validator(mode='after')
    def end_after_start(self) -> Self:
        if self.end <= self.start:
            raise ValueError("end must be after start")
        return self

book = Booking(
    start='2026-01-01',
    end='2026-01-03',
    guests=2,
    tags="beach, sun"
)

print(book)

try:
     Booking(start="2026-01-05", end="2026-01-03", guests=0)
except ValidationError as e:
    for err in e.errors(): print(err["loc"], "->", err["msg"])

start=datetime.date(2026, 1, 1) end=datetime.date(2026, 1, 3) guests=2 tags=['beach', 'sun']
('guests',) -> Value error, need atleast one guest


## 6 · Enums and `Literal` — a closed list of allowed values
- `Enums` : Fixed set of named constants
- `Literal` : Fixed raw value written directly

In [8]:
from enum import StrEnum
from typing import Literal 

class Priority(StrEnum): 
    LOW = "low"
    HIGH = "high"

class Task(BaseModel):
    priority: Priority = Priority.LOW 
    kind: Literal['bug', 'feature']

t = Task(
    priority='high',
    kind='bug'
)

print(t.priority, t.priority is Priority.HIGH)

try:
    Task(priority='urgent', kind='bug')
except ValidationError as e:
    print(e.errors()[0]['msg'])

high True
Input should be 'low' or 'high'


## 7 · Serialization — getting data back out

`Serialization` means dumping, converting the model instance, into python dictionaries and JSON encoded strings.

The single most useful trick for **PATCH** endpoints is `exclude_unset`: it tells "field not sent" apart from "field sent as null".

In [9]:
class TicketUpdate(BaseModel):
    title: str | None = None 
    priority: str | None = None 

body = TicketUpdate(priority="high")
print(body.model_dump(),'\n')

print(body.model_dump(exclude_unset=True),'\n')
# Exclude fields that were not explicitly set

print(body.model_dump_json(exclude_none=True))
# Exclude fields that were None

{'title': None, 'priority': 'high'} 

{'priority': 'high'} 

{"priority":"high"}


`ConfigDict`: Configures how Pydantic model behaves. We can control:
- Whether `Pydantic` model converts the data types automatically.
- Whether extra `Fields` are allowed.
- `Fields` can be modified creating an object.
- Whether `Pydantic` validates when you update an attribute.

In [10]:
from pydantic import ConfigDict
from pydantic.alias_generators import to_camel # converts snake_case to camelCase

class ApiUser(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,
        populate_by_name=True, 
        # Allows to create both python field names and their generated aliases
        frozen=True
        # frozen prevents modifying attributes after the object is created (not deep immutability)
    )

    first_name: str 
    created_at: datetime 

user = ApiUser(
    first_name="James",
    created_at="2026-01-01T10:00:00Z"
)

print(user)

user = ApiUser(
    firstName="James",
    createdAt="2026-01-01T10:00:00Z"
)
print(user)

u = ApiUser.model_validate(
    {
        'firstName': 'James',
        'createdAt': "2026-01-01T10:00:00Z"
    }
)
print(u.model_dump(by_alias=True, mode='json')) # camelCase output, JSON safe values
try:
    u.first_name = 'x'
except ValidationError as e:
    print("Frozen:", e.errors()[0]['type'])

first_name='James' created_at=datetime.datetime(2026, 1, 1, 10, 0, tzinfo=TzInfo(0))
first_name='James' created_at=datetime.datetime(2026, 1, 1, 10, 0, tzinfo=TzInfo(0))
{'firstName': 'James', 'createdAt': '2026-01-01T10:00:00Z'}
Frozen: frozen_instance


## 8 · ORM mode (`from_attributes`) — a response model is also a **filter**
Database objects often hold fields you must never send to a client (like password hashes).
A response model only lets the fields you list through.

**ORM mode** allows `Pydantic` read data directly from the db object instead of requiring a Python dictionary.

In [11]:
from pydantic import ConfigDict

class DbRow:
    def __init__(self):
        self.id = 1
        self.name = "James"
        self.password_hash = "$argon2id$..."

class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    name: str 

print(UserOut.model_validate(DbRow()))

id=1 name='James'


## 9 · Generics — one `Page[T]` for every list endpoint

Generics in Pydantic allow you to create flexible, reusable data models where the exact type of a field isn't fixed until you actually use the model.

In [12]:
from pydantic import BaseModel 
from typing import Generic, TypeVar

# TypeVar is a placeholder (temporary value reserved for a spot in the code to be filled later)

class User(BaseModel):
    id: int 
    username: str 

class Product(BaseModel):
    sku: str
    price: float

T = TypeVar('T')

class ApiResponse(BaseModel, Generic[T]):
    status: str 
    message: str 
    data: T

user_payload = {
    "status": "success",
    "message": "User found",
    "data": {"id": 1, "username": "alex_dev"}
}

user_response = ApiResponse[User](**user_payload)
print(user_response.data.username)
print(user_response.data)

alex_dev
id=1 username='alex_dev'


In [13]:
import math 
from pydantic import computed_field


class User(BaseModel):
    id: int
    name: str
    is_active: bool = True

class Page(BaseModel, Generic[T]):
    items: list[T]
    total: int
    size: int 

    @computed_field
    @property
    def pages(self) -> int:
        return max(1, math.ceil(self.total/self.size))

In [14]:
page = Page(items=[1,3,4],total=8,size=10)
page

Page(items=[1, 3, 4], total=8, size=10, pages=1)

In [15]:
p = Page[User](items=[User(id=1, name="Ada")], total=41, size=20)
# REmember T refers to users we need to define that
print(p.model_dump())
try: Page[User](items=[{"id": "x"}], total=1, size=1)
except ValidationError as e: print("items are validated as User:", e.errors()[0]["loc"])

{'items': [{'id': 1, 'name': 'Ada', 'is_active': True}], 'total': 41, 'size': 20, 'pages': 3}
items are validated as User: ('items', 0, 'id')


## 10 · Discriminated unions — "it depends on the `kind` field"
`Discriminator` field means its a shared property across multiple models in a Union

`TypeAdapter` lets us validate, parse and serialize in python types without creating `BaseModel` class.

In [16]:
from pydantic import TypeAdapter, BaseModel
from typing import Annotated

class Cat(BaseModel):
    kind: Literal['cat']
    lives: int 

class Dog(BaseModel):
    kind: Literal['dog']
    good_boy: bool 

Pet = Annotated[Cat | Dog, Field(discriminator='kind')]
pets = TypeAdapter(list[Pet]).validate_python([{'kind':'cat', 'lives': 9}, {"kind": "dog", "good_boy": True}])
pets

[Cat(kind='cat', lives=9), Dog(kind='dog', good_boy=True)]

## 11 · Secrets stay secret
`SecretStr` is a data type in Pydantic used to hide sensitive text e.g, passwords, API keys.

In [18]:
from pydantic import SecretStr

class Account(BaseModel):
    username: str 
    password: SecretStr

account = Account(username='james', password='op@!3wx')
print(account)
print(account.password.get_secret_value())
print(account.model_dump_json())

username='james' password=SecretStr('**********')
op@!3wx
{"username":"james","password":"**********"}
